# Retail store inventory

link: https://www.kaggle.com/datasets/anirudhchauhan/retail-store-inventory-forecasting-dataset

In [1]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

In [2]:
def TimeSeriesChart(real_data, predictions, title="Backtesting (Actual vs Forecast)"):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=real_data.index,
            y=real_data[TARGET],
            name="Actual",
            mode="lines",
            line=dict(color="#1f77b4", width=2),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=predictions.index,
            y=predictions["mean"],
            name="Forecast (Mean)",
            mode="lines",
            line=dict(color="#ff7f0e", width=2, dash="dash"),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=predictions.index,
            y=predictions["0.5"],
            name="Forecast (Median)",
            mode="lines",
            line=dict(color="#000000", width=2, dash="dash"),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=predictions.index,
            y=predictions["0.975"],
            name="Upper Bound (p97.5)",
            mode="lines",
            line=dict(width=0),
            showlegend=False,
        )
    )

    fig.add_trace(
        go.Scatter(
            x=predictions.index,
            y=predictions["0.025"],
            name="95% Confidence Interval (p2.5 - p97.5)",
            mode="lines",
            fill="tonexty",
            fillcolor="rgba(255, 127, 14, 0.15)",
            line=dict(width=0),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=predictions.index,
            y=predictions["0.95"],
            name="Upper Bound (p95)",
            mode="lines",
            line=dict(width=0),
            showlegend=False,
        )
    )

    fig.add_trace(
        go.Scatter(
            x=predictions.index,
            y=predictions["0.05"],
            name="90% Confidence Interval (p5 - p95)",
            mode="lines",
            fill="tonexty",
            fillcolor="rgba(200, 100, 14, 0.15)",
            line=dict(width=0),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=predictions.index,
            y=predictions["0.9"],
            name="Upper Bound (p90)",
            mode="lines",
            line=dict(width=0),
            showlegend=False,
        )
    )

    fig.add_trace(
        go.Scatter(
            x=predictions.index,
            y=predictions["0.1"],
            name="80% Confidence Interval (p10 - p90)",
            mode="lines",
            fill="tonexty",
            fillcolor="rgba(100, 60, 14, 0.15)",
            line=dict(width=0),
        )
    )

    fig.add_layout_image(
        dict(
            source="https://raw.githubusercontent.com/CezarT/ai_model_evaluation_mlops/main/docs/images/logo.png",
            xref="paper", yref="paper",
            x=1.05, y=1.05,
            sizex=0.2, sizey=0.2,
            xanchor="right", yanchor="bottom"
        )
    )

    fig.update_layout(
        title=f"{title}",
        xaxis_title="Date",
        yaxis_title=f"{TARGET}",
        template="plotly_white",
        hovermode="x unified",
    )
    fig.show()

In [3]:
df = pd.read_csv("../data/01_raw/retail_store_inventory.csv")
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
df["item_id"] = df["Store ID"] + "_" + df["Product ID"]
df

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,item_id
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn,S001_P0001
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn,S001_P0002
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer,S001_P0003
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn,S001_P0004
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer,S001_P0005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73095,2024-01-01,S005,P0016,Furniture,East,96,8,127,18.46,73.73,20,Snowy,0,72.45,Winter,S005_P0016
73096,2024-01-01,S005,P0017,Toys,North,313,51,101,48.43,82.57,10,Cloudy,0,83.78,Autumn,S005_P0017
73097,2024-01-01,S005,P0018,Clothing,West,278,36,151,39.65,11.11,10,Rainy,0,10.91,Winter,S005_P0018
73098,2024-01-01,S005,P0019,Toys,East,374,264,21,270.52,53.14,20,Rainy,0,55.80,Spring,S005_P0019


In [4]:
QUANTILE_LEVELS = [0.025, 0.05, 0.10, 0.50, 0.90, 0.95, 0.975]
TARGET = "Units Sold"
EVAL_METRIC = "WAPE"
KNOWN_COVARIATES_NAMES=["Category", "Region", "Weather Condition", "Holiday/Promotion", "Seasonality"]
MODELS = {
    "RecursiveTabular": {"tabular_hyperparameters": {"GBM": {}, "XGB": {}, "CAT": {}}},
    "Naive": {},
    "SeasonalNaive": {},
    "PatchTST": {},
    "ETS": {},
    "ADIDA": {},
    "DLinear": {},
    "Toto": {},
    "ARIMA": {},
    "Theta": {},
    "DeepAR": {},
    "NPTS": {},
    "TemporalFusionTransformer": {},
    "Chronos2": {},
    "SimpleFeedForward": {},
    "TiDE": {},
    "WaveNet": {},
}


In [5]:
def safe_mode(x):
    if len(x) == 0: return pd.NA
    m = x.mode()
    return m.iloc[0] if len(m) > 0 else x.iloc[0]

agg_funcs = {
    'Units Sold': 'sum',
    'Category': 'first',
    'Region': 'first',
    'Inventory Level': 'mean',
    'Units Ordered': 'sum',
    'Demand Forecast': 'sum',
    'Price': 'mean',
    'Discount': 'mean',
    'Weather Condition': safe_mode,
    'Holiday/Promotion': 'max',
    'Competitor Pricing': 'mean',
    'Seasonality': safe_mode
}

## 1. Diario (Daily)

In [6]:
df_daily = df.groupby('item_id').resample('D', on='Date').agg(agg_funcs).reset_index()
df_daily = df_daily.dropna()
df_daily

,item_id,Date,Units Sold,Category,Region,Inventory Level,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,S001_P0001,2022-01-01,127,Groceries,North,231.0,55,135.47,33.50,20.0,Rainy,0,29.69,Autumn
1,S001_P0001,2022-01-02,81,Groceries,West,116.0,104,92.94,27.95,10.0,Cloudy,0,30.89,Spring
2,S001_P0001,2022-01-03,5,Electronics,West,154.0,189,5.36,62.70,20.0,Rainy,0,58.22,Winter
3,S001_P0001,2022-01-04,58,Groceries,South,85.0,193,52.87,77.88,15.0,Cloudy,1,75.99,Winter
4,S001_P0001,2022-01-05,147,Groceries,South,238.0,37,150.27,28.46,20.0,Sunny,1,29.40,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73095,S005_P0020,2023-12-28,56,Groceries,South,198.0,27,50.18,21.75,5.0,Sunny,1,25.29,Winter
73096,S005_P0020,2023-12-29,268,Clothing,East,446.0,30,267.54,85.58,20.0,Sunny,1,87.63,Summer
73097,S005_P0020,2023-12-30,149,Toys,North,251.0,181,162.92,79.48,10.0,Cloudy,1,82.69,Autumn
73098,S005_P0020,2023-12-31,40,Furniture,East,64.0,99,59.69,90.79,5.0,Snowy,1,91.67,Winter


In [7]:
PREDICTION_LENGTH_DAILY = 14

train_data_daily = TimeSeriesDataFrame.from_data_frame(
    df_daily, id_column="item_id", timestamp_column="Date"
)

train_split_daily, test_split_daily = train_data_daily.train_test_split(
    prediction_length=PREDICTION_LENGTH_DAILY
)

predictor_daily = TimeSeriesPredictor(
    prediction_length=PREDICTION_LENGTH_DAILY,
    quantile_levels=QUANTILE_LEVELS,
    target=TARGET,
    eval_metric=EVAL_METRIC,
    known_covariates_names=KNOWN_COVARIATES_NAMES
)

predictor_daily.fit(
    train_split_daily,
    hyperparameters=MODELS
)

Beginning AutoGluon training...
AutoGluon will save models to '/home/cezartdev/Documents/cezartdev/personal/ai_model_evaluation_mlops/notebooks/AutogluonModels/ag-20260518_225016'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Mar  5 00:10:35 UTC 2026
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 11.62/11.62 GB
Total GPU Memory:   Free: 11.62 GB, Allocated: 0.00 GB, Total: 11.62 GB
GPU Count:          1
Memory Avail:       22.98 GB / 31.15 GB (73.8%)
Disk Space Avail:   112.18 GB / 230.30 GB (48.7%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WAPE,
 'hyperparameters': {'ADIDA': {},
                     'ARIMA': {},
                     'Chronos2': {},
                     'DLinear': {},
                     'DeepAR': {},
                     'ETS'

In [8]:
leaderboard_daily = predictor_daily.leaderboard(train_split_daily)
leaderboard_daily

Additional data provided, testing on additional data. Resulting leaderboard will be sorted according to test score (`score_test`).


,model,score_test,score_val,pred_time_test,pred_time_val,fit_time_marginal,fit_order
0,TemporalFusionTransformer,-0.632394,-0.632394,0.104316,0.058895,57.102224,8
1,WeightedEnsemble,-0.633306,-0.629686,21.215900,19.793581,0.315005,18
2,Chronos2,-0.634001,-0.634001,4.130646,3.464523,2.553547,7
3,Toto,-0.636538,-0.631137,20.047592,18.769096,1.892918,17
4,NPTS,-0.637423,-0.637423,0.533699,0.649096,0.020418,5
5,PatchTST,-0.638129,-0.638129,0.256633,0.073333,20.080443,10
6,WaveNet,-0.638275,-0.644075,0.186965,0.174753,42.126502,12
7,TiDE,-0.642739,-0.642739,0.128122,0.112665,38.891903,11
8,DeepAR,-0.645312,-0.643161,0.220377,0.185512,66.839370,9
9,SimpleFeedForward,-0.645409,-0.645409,0.042749,0.045471,7.157626,16


In [9]:
best_model_daily = leaderboard_daily.loc[0, "model"]
predictions_daily = predictor_daily.predict(
    train_split_daily, 
    model=best_model_daily,
    known_covariates=test_split_daily.slice_by_timestep(-PREDICTION_LENGTH_DAILY, None)
)

In [10]:
selected_item = df_daily["item_id"].iloc[0]

real_data_daily = test_split_daily.loc[selected_item]
predictions_store_daily = predictions_daily.loc[selected_item]

TimeSeriesChart(real_data_daily, predictions_store_daily, title=f"Daily Backtesting - {selected_item}")

## 2. Semanal (Weekly)

In [11]:
df_weekly = df.groupby('item_id').resample('W', on='Date').agg(agg_funcs).reset_index()
df_weekly = df_weekly.dropna()
df_weekly

,item_id,Date,Units Sold,Category,Region,Inventory Level,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,S001_P0001,2022-01-02,208,Groceries,North,173.500000,159,228.41,30.725000,15.000000,Cloudy,0,30.290000,Autumn
1,S001_P0001,2022-01-09,706,Electronics,West,210.571429,977,718.59,60.300000,12.857143,Sunny,1,58.887143,Winter
2,S001_P0001,2022-01-16,686,Electronics,West,182.285714,1031,712.51,52.840000,10.000000,Snowy,1,53.482857,Winter
3,S001_P0001,2022-01-23,1142,Groceries,East,293.714286,1051,1179.51,59.672857,7.857143,Cloudy,1,59.630000,Spring
4,S001_P0001,2022-01-30,685,Toys,South,269.000000,740,747.78,65.421429,11.428571,Snowy,1,66.245714,Autumn
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10595,S005_P0020,2023-12-10,1224,Clothing,North,344.142857,502,1293.56,56.791429,8.571429,Snowy,1,56.262857,Winter
10596,S005_P0020,2023-12-17,991,Groceries,West,209.857143,843,1033.18,46.325714,13.571429,Sunny,1,47.938571,Spring
10597,S005_P0020,2023-12-24,1071,Electronics,North,346.714286,721,1077.98,58.037143,11.428571,Rainy,1,57.688571,Spring
10598,S005_P0020,2023-12-31,835,Furniture,South,244.714286,714,880.81,58.584286,5.714286,Cloudy,1,59.665714,Autumn


In [12]:
PREDICTION_LENGTH_WEEKLY = 12

train_data_weekly = TimeSeriesDataFrame.from_data_frame(
    df_weekly, id_column="item_id", timestamp_column="Date"
)

train_split_weekly, test_split_weekly = train_data_weekly.train_test_split(
    prediction_length=PREDICTION_LENGTH_WEEKLY
)

predictor_weekly = TimeSeriesPredictor(
    prediction_length=PREDICTION_LENGTH_WEEKLY,
    quantile_levels=QUANTILE_LEVELS,
    target=TARGET,
    eval_metric=EVAL_METRIC,
    known_covariates_names=KNOWN_COVARIATES_NAMES
)

predictor_weekly.fit(
    train_split_weekly,
    hyperparameters=MODELS
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260518_225541"
Beginning AutoGluon training...
AutoGluon will save models to '/home/cezartdev/Documents/cezartdev/personal/ai_model_evaluation_mlops/notebooks/AutogluonModels/ag-20260518_225541'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Mar  5 00:10:35 UTC 2026
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 11.61/11.62 GB
Total GPU Memory:   Free: 11.61 GB, Allocated: 0.01 GB, Total: 11.62 GB
GPU Count:          1
Memory Avail:       19.91 GB / 31.15 GB (63.9%)
Disk Space Avail:   112.18 GB / 230.30 GB (48.7%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WAPE,
 'hyperparameters': {'ADIDA': {},
                     'ARIMA': {},
                     'Chronos2': {},
                

In [13]:
leaderboard_weekly = predictor_weekly.leaderboard(train_split_weekly)
leaderboard_weekly

Additional data provided, testing on additional data. Resulting leaderboard will be sorted according to test score (`score_test`).


,model,score_test,score_val,pred_time_test,pred_time_val,fit_time_marginal,fit_order
0,WeightedEnsemble,-0.236677,-0.236702,0.506711,0.367441,0.328276,18
1,TemporalFusionTransformer,-0.237428,-0.237428,0.086854,0.039520,29.484569,8
2,DeepAR,-0.238204,-0.238925,0.170107,0.124225,21.530617,9
3,SimpleFeedForward,-0.239754,-0.239754,0.048655,0.037385,20.451478,16
4,Chronos2,-0.239816,-0.239816,1.484255,0.582587,0.913587,7
5,TiDE,-0.240215,-0.240215,0.087143,0.072956,43.931222,11
6,WaveNet,-0.240590,-0.240302,0.112285,0.091352,18.268632,12
7,ETS,-0.240689,-0.240689,0.069238,0.069287,0.012496,4
8,ARIMA,-0.244283,-0.244283,0.253732,0.249483,0.011082,15
9,PatchTST,-0.244716,-0.244716,0.065224,0.050540,17.595427,10


In [14]:
best_model_weekly = leaderboard_weekly.loc[0, "model"]
predictions_weekly = predictor_weekly.predict(
    train_split_weekly, 
    model=best_model_weekly,
    known_covariates=test_split_weekly.slice_by_timestep(-PREDICTION_LENGTH_WEEKLY, None)
)

In [15]:
real_data_weekly = test_split_weekly.loc[selected_item]
predictions_store_weekly = predictions_weekly.loc[selected_item]

TimeSeriesChart(real_data_weekly, predictions_store_weekly, title=f"Weekly Backtesting - {selected_item}")

## 3. Mensual (Monthly)

In [16]:
df_monthly = df.groupby('item_id').resample('ME', on='Date').agg(agg_funcs).reset_index()
df_monthly = df_monthly.dropna()
df_monthly

,item_id,Date,Units Sold,Category,Region,Inventory Level,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,S001_P0001,2022-01-31,3627,Groceries,North,240.193548,4110,3799.04,58.060323,10.483871,Snowy,1,57.973548,Winter
1,S001_P0001,2022-02-28,4100,Groceries,East,269.178571,3190,4264.98,52.272143,9.285714,Rainy,1,52.537500,Spring
2,S001_P0001,2022-03-31,3669,Toys,North,271.548387,3510,3848.34,53.409677,9.516129,Snowy,1,53.446452,Autumn
3,S001_P0001,2022-04-30,4874,Furniture,West,276.633333,3315,4978.40,58.679000,10.833333,Cloudy,1,59.140000,Winter
4,S001_P0001,2022-05-31,3554,Electronics,East,259.000000,3391,3702.70,47.627419,10.000000,Sunny,1,47.816129,Spring
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2495,S005_P0020,2023-09-30,4471,Groceries,South,290.833333,3272,4576.83,50.806000,9.166667,Cloudy,1,51.778333,Winter
2496,S005_P0020,2023-10-31,3936,Clothing,West,288.387097,4056,4090.56,60.454516,9.838710,Cloudy,1,60.883871,Spring
2497,S005_P0020,2023-11-30,4894,Clothing,West,330.533333,2963,5029.48,56.616333,11.000000,Rainy,1,56.698333,Winter
2498,S005_P0020,2023-12-31,4509,Groceries,South,279.709677,3123,4694.56,56.706452,9.838710,Sunny,1,57.016774,Spring


In [17]:
PREDICTION_LENGTH_MONTHLY = 6

train_data_monthly = TimeSeriesDataFrame.from_data_frame(
    df_monthly, id_column="item_id", timestamp_column="Date"
)

train_split_monthly, test_split_monthly = train_data_monthly.train_test_split(
    prediction_length=PREDICTION_LENGTH_MONTHLY
)

predictor_monthly = TimeSeriesPredictor(
    prediction_length=PREDICTION_LENGTH_MONTHLY,
    quantile_levels=QUANTILE_LEVELS,
    target=TARGET,
    eval_metric=EVAL_METRIC,
    known_covariates_names=KNOWN_COVARIATES_NAMES
)

predictor_monthly.fit(
    train_split_monthly,
    hyperparameters=MODELS
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260518_225844"
Beginning AutoGluon training...
AutoGluon will save models to '/home/cezartdev/Documents/cezartdev/personal/ai_model_evaluation_mlops/notebooks/AutogluonModels/ag-20260518_225844'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Mar  5 00:10:35 UTC 2026
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 11.61/11.62 GB
Total GPU Memory:   Free: 11.61 GB, Allocated: 0.01 GB, Total: 11.62 GB
GPU Count:          1
Memory Avail:       19.37 GB / 31.15 GB (62.2%)
Disk Space Avail:   112.15 GB / 230.30 GB (48.7%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WAPE,
 'hyperparameters': {'ADIDA': {},
                     'ARIMA': {},
                     'Chronos2': {},
                

In [18]:
leaderboard_monthly = predictor_monthly.leaderboard(train_split_monthly)
leaderboard_monthly

Additional data provided, testing on additional data. Resulting leaderboard will be sorted according to test score (`score_test`).


,model,score_test,score_val,pred_time_test,pred_time_val,fit_time_marginal,fit_order
0,WeightedEnsemble,-0.123575,-0.123585,4.564973,2.909601,0.283175,18
1,TemporalFusionTransformer,-0.124767,-0.124767,0.076368,0.035995,25.502007,8
2,Chronos2,-0.127986,-0.127986,1.184878,0.312730,0.909144,7
3,ETS,-0.130215,-0.130215,0.038376,0.047583,0.011827,4
4,ADIDA,-0.130647,-0.130647,0.048262,0.053537,0.011675,13
5,WaveNet,-0.132027,-0.131816,0.080808,0.065158,15.265344,12
6,DeepAR,-0.136014,-0.137044,0.087164,0.069545,38.464406,9
7,TiDE,-0.136954,-0.136954,0.076120,0.068754,25.960535,11
8,Toto,-0.138030,-0.138664,4.262697,2.661234,1.605042,17
9,Theta,-0.140003,-0.140003,0.050400,0.047179,0.010991,6


In [19]:
best_model_monthly = leaderboard_monthly.loc[0, "model"]
predictions_monthly = predictor_monthly.predict(
    train_split_monthly, 
    model=best_model_monthly,
    known_covariates=test_split_monthly.slice_by_timestep(-PREDICTION_LENGTH_MONTHLY, None)
)

In [20]:
real_data_monthly = test_split_monthly.loc[selected_item]
predictions_store_monthly = predictions_monthly.loc[selected_item]

TimeSeriesChart(real_data_monthly, predictions_store_monthly, title=f"Monthly Backtesting - {selected_item}")